# ai-detector trên Kaggle — phát hiện giọng nói giả tiếng Việt

Notebook này chạy **trọn pipeline** trên GPU miễn phí của Kaggle:

```
REAL (dataset tiếng Việt) → FAKE (Piper · Kokoro · OmniVoice) → augment → WavLM → Classifier
```

## Trước khi chạy, bật hai thứ trong panel bên phải

| Mục | Đặt thành | Vì sao |
|---|---|---|
| **Accelerator** | `GPU T4 x2` hoặc `GPU P100` | OmniVoice (voice cloning) gần như không chạy được trên CPU |
| **Internet** | `On` | cần tải WavLM, giọng Piper/Kokoro và cài thư viện |

Rồi **Add Input → Datasets** một bộ giọng thật tiếng Việt (VIVOS, Common Voice vi, …).
Pipeline tự nhận diện định dạng nên không cần chỉnh gì thêm.

> **Phiên Kaggle ~9 giờ và xoá sạch khi kết thúc.** Ô cuối cùng đóng gói corpus +
> checkpoint vào `/kaggle/working` để tải về hoặc lưu thành Kaggle Dataset dùng lại
> cho phiên sau.

## 1. Kiểm tra máy

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "KHÔNG có GPU — hãy bật Accelerator"
!df -h /kaggle/working | tail -1
!ls /kaggle/input 2>/dev/null || echo "Chưa add dataset nào"

## 2. Lấy code

Hai cách, chọn một:
- **Clone từ GitHub** (mặc định) — cần Internet: On.
- **Dùng Kaggle Dataset chứa code** — nếu repo private: upload thư mục dự án thành
  một Dataset rồi đặt `CODE_DATASET` trỏ tới nó.

In [ ]:
import os, shutil, sys
from pathlib import Path

REPO = "https://github.com/<tài-khoản>/ai-detector.git"   # ← sửa thành repo của bạn
CODE_DATASET = None      # ← hoặc: "/kaggle/input/ai-detector-code"
WORK = Path("/kaggle/working/ai-detector")

if CODE_DATASET:
    if WORK.exists():
        shutil.rmtree(WORK)
    shutil.copytree(CODE_DATASET, WORK)
elif not WORK.exists():
    !git clone --depth 1 {REPO} {WORK}

os.chdir(WORK)
sys.path.insert(0, str(WORK))
print("Thư mục làm việc:", Path.cwd())
!ls

## 3. Cài thư viện

Kaggle đã có sẵn torch + CUDA, nên chỉ cài phần còn thiếu. Engine sinh fake cài
riêng — cái nào lỗi thì bỏ qua, `info` bên dưới sẽ cho biết cái nào dùng được.

In [ ]:
!pip install -q -r requirements.txt

# Engine sinh audio giả (cài lần lượt, lỗi cái nào bỏ cái đó)
!pip install -q piper-tts                                                        || true
!pip install -q git+https://github.com/iamdinhthuan/Kokoro-Vietnamese.git        || true
!pip install -q omnivoice                                                        || true

!apt-get -qq install -y ffmpeg > /dev/null 2>&1 || true   # cần cho augment MP3/AAC

In [ ]:
CFG = "configs/kaggle.yaml"
!python -m aidetector info -c {CFG}

## 4. REAL dataset

`ingest` tự nhận diện loại dataset (VIVOS / Common Voice / thư mục wav / real+fake
chia sẵn) rồi ép mọi file về chuẩn corpus: 16 kHz mono PCM_16, 3–10 giây, chuẩn mức
âm lượng, cắt im lặng, loại clipping/NaN.

Sửa `RAW` cho khớp dataset bạn vừa add.

In [ ]:
RAW = "/kaggle/input/vivos"      # ← đường dẫn dataset đã add
LIMIT = 4000                     # số utterance tối đa; bỏ đi nếu muốn lấy hết

!python -m aidetector ingest {RAW} -c {CFG} --limit {LIMIT} --per-speaker 120

In [ ]:
!python -m aidetector validate -c {CFG}

## 5. FAKE dataset

Mỗi audio giả được sinh từ **chính transcript và speaker của một utterance thật**,
nên luôn có bản real đối chứng cùng nội dung, cùng giọng. OmniVoice còn clone thẳng
giọng của speaker đó từ một câu khác của họ.

Càng nhiều engine, mô hình càng khó "học vẹt" theo dấu vết của một engine duy nhất.

In [ ]:
# Chia đều số lượng cho các engine đã cài được. Bỏ --engines để dùng danh sách
# trong configs/kaggle.yaml (piper, kokoro, omnivoice).
!python -m aidetector generate -c {CFG} --engines piper kokoro --count 1200

In [ ]:
# OmniVoice chậm hơn nhiều (voice cloning) — chạy riêng để dễ theo dõi và
# dễ dừng giữa chừng mà không mất phần đã sinh.
!python -m aidetector generate -c {CFG} --engines omnivoice --count 800 --device cuda

## 6. Chia tập → augment

`split` chạy **trước** `augment`: bản augment chỉ sinh cho train và bám đúng split
của bản gốc, còn val/test giữ audio sạch để số đo phản ánh dữ liệu thật.

Thêm `--holdout omnivoice` nếu muốn giữ hẳn một engine riêng cho test — đó là phép
đo sát thực tế nhất: mô hình có bắt được engine **chưa từng thấy** hay không.

In [ ]:
!python -m aidetector split   -c {CFG}
!python -m aidetector augment -c {CFG} --copies 1

## 7. WavLM → Classifier

Embedding được cache theo `utt_id` nên chạy lại chỉ trích phần mới. Đổi backbone chỉ
cần `--set features.backbone.name=wav2vec2` (cache tách riêng, không đè lên nhau).

In [ ]:
!python -m aidetector features -c {CFG}
!python -m aidetector train    -c {CFG}
!python -m aidetector evaluate -c {CFG}

## 8. Kết quả

In [ ]:
import json
from pathlib import Path
from IPython.display import Image, display

metrics = json.loads(Path("/kaggle/working/reports/metrics.json").read_text())
overall = metrics["overall"]
print(f"EER      : {overall['eer'] * 100:.2f}%      ← số đo chính")
print(f"ROC-AUC  : {overall['roc_auc']:.4f}")
print(f"min-DCF  : {overall['min_dcf']:.4f}")
print(f"Accuracy : {overall['accuracy'] * 100:.2f}%  (ngưỡng {overall['threshold']:.3f})")

print("\nTheo từng generator:")
for name, entry in metrics["by_generator"].items():
    if "eer_vs_all_real" in entry:
        print(f"  {name:<42} n={entry['n']:>5} · EER {entry['eer_vs_all_real'] * 100:6.2f}%"
              f" · bắt được {entry['detection_rate'] * 100:5.1f}%")
    elif "false_alarm_rate" in entry:
        print(f"  {name:<42} n={entry['n']:>5} · báo nhầm {entry['false_alarm_rate'] * 100:5.1f}%")

print("\nClean vs augmented:")
for name, entry in metrics["by_condition"].items():
    print(f"  {name:<12} n={entry['n']:>5} · điểm trung bình {entry['mean_score']:.3f}")

display(Image("/kaggle/working/reports/curves.png"))
display(Image("/kaggle/working/reports/confusion_matrix.png"))

## 9. Thử trên một file bất kỳ

In [ ]:
!python -m aidetector detect -c {CFG} /kaggle/working/corpus/audio/fake/piper/*/*.wav --json | head -30

## 10. Lưu lại trước khi hết phiên

`/kaggle/working` bị xoá khi phiên kết thúc, và commit output với hàng chục nghìn
file wav rời rạc thì rất chậm — nên gói corpus vào **một** file zip.

Sau khi notebook chạy xong: **Output → New Dataset**. Phiên sau chỉ cần add dataset
đó rồi chạy ô "Phiên tiếp theo" ở dưới, khỏi phải ingest và generate lại.

In [ ]:
!python -m aidetector pack -c {CFG} --out /kaggle/working/corpus.zip

import shutil
shutil.make_archive("/kaggle/working/model", "zip", "/kaggle/working/checkpoints")
shutil.make_archive("/kaggle/working/reports_bundle", "zip", "/kaggle/working/reports")

!ls -lh /kaggle/working/*.zip

### Phiên tiếp theo — bỏ qua ingest/generate

Add dataset chứa `corpus.zip` vừa tạo, rồi chạy:

```python
!python -m aidetector unpack /kaggle/input/<tên-dataset>/corpus.zip -c configs/kaggle.yaml
!python -m aidetector run split augment features train evaluate -c configs/kaggle.yaml
```

### Vài nút chỉnh hay dùng

```python
# Đổi backbone (cache đặc trưng tách riêng nên không đụng nhau)
!python -m aidetector run features train evaluate -c {CFG} \
        --set features.backbone.name=wav2vec2

# Đo khả năng tổng quát sang engine chưa từng thấy
!python -m aidetector split -c {CFG} --holdout omnivoice
!python -m aidetector run features train evaluate -c {CFG}

# Augment mạnh tay hơn nếu clean và augmented chênh lệch nhiều
!python -m aidetector augment -c {CFG} --copies 3 --set augment.ops.codec.p=0.8
```